# RAG Agent Demo

This notebook creates a temporary Azure AI Foundry agent that uses the Azure AI Search index created earlier.

The agent answers a question using the synthetic documents.

The agent is deleted at the end.

## Setup

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ConnectionType
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import AzureAISearchTool, MessageRole

# The notebook is inside walkthrough/, so .env is one folder above it.
load_dotenv(dotenv_path=Path("../.env"))

AIPROJECT_ENDPOINT = os.getenv("AIPROJECT_ENDPOINT")
CHAT_MODEL = os.getenv("CHAT_MODEL")
AZURE_AI_SEARCH_INDEX = os.getenv("AZURE_AI_SEARCH_INDEX")

## Create clients

In [2]:
credential = DefaultAzureCredential()

project_client = AIProjectClient(
    endpoint=AIPROJECT_ENDPOINT,
    credential=credential
)

agents_client = AgentsClient(
    endpoint=AIPROJECT_ENDPOINT,
    credential=credential
)

print("Project client created.")
print("Agents client created.")

Project client created.
Agents client created.


## Get the Azure AI Search connection

In [3]:
# Get the Azure AI Search connection that was added to the Foundry project earlier.

search_connection = project_client.connections.get_default(
    connection_type=ConnectionType.AZURE_AI_SEARCH
)

print("Search connection type:", search_connection.type)
print("Search connection found:", bool(search_connection.id))

Search connection type: ConnectionType.AZURE_AI_SEARCH
Search connection found: True


## Create the Search tool

In [4]:
# This tool tells the agent which Azure AI Search index to use.

ai_search = AzureAISearchTool(
    index_connection_id=search_connection.id,
    index_name=AZURE_AI_SEARCH_INDEX,
    top_k=5
)

print("Azure AI Search tool created.")

Azure AI Search tool created.


## Create a temporary agent

In [5]:
agent = agents_client.create_agent(
    model=CHAT_MODEL,
    name="temporary-rag-demo-agent",
    instructions=(
        "You answer questions using the connected Azure AI Search index. "
        "Use the retrieved information when available. "
        "If the answer is not available in the indexed documents, say that the evidence does not specify it."
    ),
    tools=ai_search.definitions,
    tool_resources=ai_search.resources
)

print("Temporary agent created.")
print("Agent ID:", agent.id)

Temporary agent created.
Agent ID: asst_PsylBWQ73YGBRuObplbCNemF


## Ask a question

In [6]:
question = "Does AcmeCloud Analytics encrypt customer data at rest?"

thread = agents_client.threads.create()

agents_client.messages.create(
    thread_id=thread.id,
    role="user",
    content=question
)

run = agents_client.runs.create_and_process(
    thread_id=thread.id,
    agent_id=agent.id
)

print("Question:", question)
print("Run status:", run.status)

Question: Does AcmeCloud Analytics encrypt customer data at rest?
Run status: RunStatus.COMPLETED


## Print the answer

In [7]:
# Get the latest assistant message from the thread.

assistant_message = agents_client.messages.get_last_message_by_role(
    thread_id=thread.id,
    role=MessageRole.AGENT
)

print(assistant_message)

{'id': 'msg_25VWsvptByPFvTRMffczpc7b', 'object': 'thread.message', 'created_at': 1781472918, 'assistant_id': 'asst_PsylBWQ73YGBRuObplbCNemF', 'thread_id': 'thread_UxGEXgnw5KTS82PoOB8Kyx06', 'run_id': 'run_aoKJiAnYVbwQC85zmf6tSNAD', 'role': 'assistant', 'content': [{'type': 'text', 'text': {'value': 'Yes, AcmeCloud Analytics encrypts customer data at rest using AES-256. This encryption covers production application databases, object storage for report assets, and internal storage used by the analytics processing layer. Additionally, backups of customer data are also encrypted, including scheduled production backups, restoration test copies, and archived backup snapshots. These backup encryption settings are reviewed regularly during quarterly backup restoration tests【3:1†Data Retention Policy】.', 'annotations': [{'type': 'url_citation', 'text': '【3:1†Data Retention Policy】', 'start_index': 475, 'end_index': 502, 'url_citation': {'url': 'doc_1', 'title': 'encryption_policy.md'}}]}}], 'at

In [8]:
print(assistant_message["content"][0]["text"]["value"])

Yes, AcmeCloud Analytics encrypts customer data at rest using AES-256. This encryption covers production application databases, object storage for report assets, and internal storage used by the analytics processing layer. Additionally, backups of customer data are also encrypted, including scheduled production backups, restoration test copies, and archived backup snapshots. These backup encryption settings are reviewed regularly during quarterly backup restoration tests【3:1†Data Retention Policy】.


## Delete the temporary agent

In [9]:
agents_client.delete_agent(agent.id)

print("Temporary agent deleted.")

Temporary agent deleted.
